# TP1 — G4 — Baseline clássico para o SIIM-FISABIO-RSNA COVID-19 Detection (2021)

Este notebook reproduz **todas** as tabelas e figuras do artigo.

**Configuração no Kaggle (painel direito):**
1. *Add Input* → aba *Competitions* → `SIIM-FISABIO-RSNA COVID-19 Detection` (é preciso ter aceitado as regras da competição).
2. *Settings* → *Internet*: **On** (exige conta verificada por telefone).
3. *Accelerator*: **None** (CPU; o método não usa GPU).

**Execução recomendada:**
- Teste rápido: defina `SAMPLE_FRAC = 0.1` e rode célula a célula (~20–30 min).
- Execução final: volte para `SAMPLE_FRAC = 1.0` e use *Save Version → Save & Run All (Commit)*. Roda em segundo plano (limite de 12 h), e os arquivos de `/kaggle/working` ficam disponíveis na aba *Output*.

In [ ]:
REPO_URL = "https://github.com/FelipeFreires-Costa/COVID-19-AI-Detection-AP1.git"
SAMPLE_FRAC = 1.0  # 0.1 para teste rapido

import glob, os
os.chdir("/kaggle/working")
if not os.path.exists("repo"):
    os.system(f"git clone -q {REPO_URL} repo")
os.chdir("/kaggle/working/repo")

csv = glob.glob("/kaggle/input/**/train_study_level.csv", recursive=True)
assert csv, "Adicione a competicao siim-covid19-detection como input do notebook"
os.environ["SIIM_DATA_DIR"] = os.path.dirname(csv[0])
os.environ["TP1_WORK_DIR"] = "/kaggle/temp/work" if os.path.isdir("/kaggle/temp") else "/kaggle/working/work"
os.environ["TP1_SAMPLE_FRAC"] = str(SAMPLE_FRAC)
RES = "results" if SAMPLE_FRAC >= 1 else f"results_frac{SAMPLE_FRAC:g}"
print(os.environ["SIIM_DATA_DIR"], RES)

In [ ]:
# Decodificadores de DICOM comprimido (JPEG/JPEG-LS/JPEG2000).
!pip install -q pylibjpeg pylibjpeg-libjpeg pylibjpeg-openjpeg python-gdcm
!mkdir -p $RES && pip freeze > $RES/environment_kaggle.txt
!python -c "import sklearn, skimage, pydicom, lightgbm, cv2; print(sklearn.__version__, skimage.__version__, pydicom.__version__, lightgbm.__version__, cv2.__version__)"

## 1. Dados: DICOM → PNG 512, metadados, amostragem e partição por paciente (Felipe)

In [ ]:
!python src/prepare_data.py

## 2. Análise exploratória (Felipe)

In [ ]:
!python src/eda.py
from IPython.display import Image, display
display(Image(f"{RES}/figures/eda_overview.png"))
display(Image(f"{RES}/figures/eda_examples.png", width=700))

## 3. Pré-processamento e extração de características (Eduardo)

In [ ]:
!python src/extract_features.py --tag default --clahe 1 --mask 1
!python src/make_figures.py --only masks
display(Image(f"{RES}/figures/fig_masks.png"))

## 4. Grade descritor × modelo com CV aninhada + avaliação no hold-out (Pedro)

In [ ]:
!python src/run_experiments.py --tag default
import pandas as pd
s = pd.read_csv(f"{RES}/tables/cv_summary.csv")
s[["featureset", "model", "auc_macro_mean", "auc_macro_std", "bal_acc_mean", "f1_macro_mean", "bin_auc_mean"]].sort_values("auc_macro_mean", ascending=False)

## 5. Ablações: pré-processamento (Eduardo) e desbalanceamento (Pedro)

In [ ]:
!python src/extract_features.py --tag raw_full --clahe 0 --mask 0
!python src/extract_features.py --tag clahe_full --clahe 1 --mask 0
!python src/extract_features.py --tag raw_mask --clahe 0 --mask 1
!python src/ablation.py --part all
display(pd.read_csv(f"{RES}/tables/ablation_prep.csv"))
display(pd.read_csv(f"{RES}/tables/ablation_imb.csv"))

## 6. Figuras e análise de erro (Pedro)

In [ ]:
!python src/make_figures.py
for f in ["fig_roc_cm.png", "fig_importance.png", "fig_errors.png"]:
    display(Image(f"{RES}/figures/{f}", width=800))
display(pd.read_csv(f"{RES}/tables/error_by_group.csv"))

## 7. Preencher o artigo com os resultados
Gera `paper/numbers.tex`, copia tabelas e figura para `paper/` e cria `TP1_G4_paper_overleaf.zip` (Overleaf → *New Project* → *Upload Project*).

In [ ]:
!python src/fill_paper.py
print(open("paper/numbers.tex").read())

## 8. Empacotar tudo para o GitHub
Baixe `tp1_results.zip` na aba *Output*, descompacte por cima da pasta do repositório e faça commit de `results/`, `splits/` e `paper/`.

In [ ]:
!cd /kaggle/working/repo && zip -qr /kaggle/working/tp1_results.zip $RES splits paper
!cp $RES/TP1_G4_paper_overleaf.zip /kaggle/working/ 2>/dev/null; ls -la /kaggle/working